# Yearly Park Factors by Team, Day/Night, and Overall

This notebook calculates a run-based park factor for every team-year combination in the project.

Definition used in this notebook:

- `park_factor = 100 * (home total runs per game) / (road total runs per game)`
- `overall`: all games in that season
- `day`: only games with local venue `start_hour < 17`
- `night`: only games with local venue `start_hour >= 17`

Notes:

- A value above `100` means that park played more hitter-friendly than the team's road games for that split.
- A value below `100` means that park played more pitcher-friendly than the team's road games for that split.
- The day/night split is based on the local time of the ballpark where the game was played.
- This notebook uses `master_data.csv` so it can correctly classify both home and road games.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

# ============================================================
# PATHS / PARAMETERS
# ============================================================
TEAM_PARAMS_PATH = 'team_parameters.csv'
MASTER_DATA_PATH = os.path.join('..', 'data', 'master_data.csv')

OUTPUT_LONG_PATH = os.path.join('..', 'data', 'park_factor_team_year_day_night_long.csv')
OUTPUT_WIDE_PATH = os.path.join('..', 'data', 'park_factor_team_year_day_night_wide.csv')

DAY_START_CUTOFF_HOUR = 17

print(f'Team parameters: {os.path.abspath(TEAM_PARAMS_PATH)}')
print(f'Master data:     {os.path.abspath(MASTER_DATA_PATH)}')
print(f'Output (long):   {os.path.abspath(OUTPUT_LONG_PATH)}')
print(f'Output (wide):   {os.path.abspath(OUTPUT_WIDE_PATH)}')

In [ ]:
params = pd.read_csv(TEAM_PARAMS_PATH)
master = pd.read_csv(MASTER_DATA_PATH)

required_master_cols = {'game_pk', 'season', 'home_team', 'away_team', 'total_runs', 'game_start_utc'}
missing_master_cols = required_master_cols - set(master.columns)
if missing_master_cols:
    raise ValueError(f'master_data.csv is missing required columns: {sorted(missing_master_cols)}')

required_param_cols = {'team_code', 'team_name', 'stadium_name', 'data_start_year', 'data_end_year', 'timezone'}
missing_param_cols = required_param_cols - set(params.columns)
if missing_param_cols:
    raise ValueError(f'team_parameters.csv is missing required columns: {sorted(missing_param_cols)}')

master = master.copy()
master['game_start_utc'] = pd.to_datetime(master['game_start_utc'], utc=True, errors='coerce')

# Attach the timezone for the venue where the game was played.
tz_lookup = params[['team_code', 'timezone']].rename(columns={'team_code': 'home_team', 'timezone': 'venue_timezone'})
master = master.merge(tz_lookup, on='home_team', how='left')

master['start_hour_local'] = pd.Series(index=master.index, dtype='float64')
for tz_name in sorted(master['venue_timezone'].dropna().unique()):
    idx = master['venue_timezone'] == tz_name
    master.loc[idx, 'start_hour_local'] = master.loc[idx, 'game_start_utc'].dt.tz_convert(tz_name).dt.hour

master['day_night'] = np.where(master['start_hour_local'] < DAY_START_CUTOFF_HOUR, 'day', 'night')
master.loc[master['start_hour_local'].isna(), 'day_night'] = pd.NA

print(f'Loaded {len(params)} teams')
print(f'Loaded {len(master)} master game rows')
print(master[['game_pk', 'season', 'home_team', 'away_team', 'total_runs', 'start_hour_local', 'day_night']].head())

In [ ]:
records = []
splits = ['overall', 'day', 'night']

for _, team_row in params.iterrows():
    team_code = team_row['team_code']
    team_name = team_row['team_name']
    stadium_name = team_row['stadium_name']
    season_start = int(team_row['data_start_year'])
    season_end = int(team_row['data_end_year'])

    home_games_all = master[
        (master['home_team'] == team_code) &
        (master['season'] >= season_start) &
        (master['season'] <= season_end)
    ].copy()

    road_games_all = master[
        (master['away_team'] == team_code) &
        (master['season'] >= season_start) &
        (master['season'] <= season_end)
    ].copy()

    for season in range(season_start, season_end + 1):
        home_year = home_games_all[home_games_all['season'] == season].copy()
        road_year = road_games_all[road_games_all['season'] == season].copy()

        for split in splits:
            if split == 'overall':
                home_split = home_year
                road_split = road_year
            else:
                home_split = home_year[home_year['day_night'] == split].copy()
                road_split = road_year[road_year['day_night'] == split].copy()

            home_games = len(home_split)
            road_games = len(road_split)

            home_runs_per_game = home_split['total_runs'].mean() if home_games > 0 else np.nan
            road_runs_per_game = road_split['total_runs'].mean() if road_games > 0 else np.nan

            if pd.notna(home_runs_per_game) and pd.notna(road_runs_per_game) and road_runs_per_game != 0:
                park_factor = 100 * home_runs_per_game / road_runs_per_game
            else:
                park_factor = np.nan

            records.append({
                'team_code': team_code,
                'team_name': team_name,
                'stadium_name': stadium_name,
                'season': season,
                'split': split,
                'home_games': home_games,
                'road_games': road_games,
                'home_total_runs_per_game': home_runs_per_game,
                'road_total_runs_per_game': road_runs_per_game,
                'park_factor': park_factor,
            })

park_factor_long = pd.DataFrame(records)

park_factor_long[['home_total_runs_per_game', 'road_total_runs_per_game', 'park_factor']] = (
    park_factor_long[['home_total_runs_per_game', 'road_total_runs_per_game', 'park_factor']].round(3)
)

factor_wide = (
    park_factor_long
    .pivot(index=['team_code', 'team_name', 'stadium_name', 'season'], columns='split', values='park_factor')
    .reset_index()
    .rename(columns={
        'overall': 'overall_park_factor',
        'day': 'day_park_factor',
        'night': 'night_park_factor',
    })
)

home_count_wide = (
    park_factor_long
    .pivot(index=['team_code', 'team_name', 'stadium_name', 'season'], columns='split', values='home_games')
    .reset_index()
    .rename(columns={
        'overall': 'overall_home_games',
        'day': 'day_home_games',
        'night': 'night_home_games',
    })
)

road_count_wide = (
    park_factor_long
    .pivot(index=['team_code', 'team_name', 'stadium_name', 'season'], columns='split', values='road_games')
    .reset_index()
    .rename(columns={
        'overall': 'overall_road_games',
        'day': 'day_road_games',
        'night': 'night_road_games',
    })
)

park_factor_wide = (
    factor_wide
    .merge(home_count_wide, on=['team_code', 'team_name', 'stadium_name', 'season'], how='left')
    .merge(road_count_wide, on=['team_code', 'team_name', 'stadium_name', 'season'], how='left')
    .sort_values(['season', 'team_code'])
    .reset_index(drop=True)
)

print(f'Long rows: {len(park_factor_long)}')
print(f'Wide rows: {len(park_factor_wide)}')
park_factor_wide.head(15)

In [ ]:
park_factor_long.to_csv(OUTPUT_LONG_PATH, index=False)
park_factor_wide.to_csv(OUTPUT_WIDE_PATH, index=False)

print(f'Saved long results: {os.path.abspath(OUTPUT_LONG_PATH)}')
print(f'Saved wide results: {os.path.abspath(OUTPUT_WIDE_PATH)}')